# 12 — Performance Plots

Generates all paper figures from the hardcoded table data (Tables 2–5).  
No data files required — run straight through.  
All plots saved to the same folder as this notebook.

Figures produced:
1. `fig1_qps_comparison.png` — QPS bar chart, full scale
2. `fig2_recall_vs_k.png` — Recall@k vs reranking depth K
3. `fig3_memory_comparison.png` — Memory footprint (vector + index)
4. `fig4_recall_qps_tradeoff.png` — Recall@100 vs QPS scatter (Pareto)
5. `fig5_adaptive_k.png` — Adaptive-K: recall vs average candidate budget

In [1]:
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

OUT = Path('.').resolve()
print(f"Saving plots to: {OUT}")

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

C_BASE    = '#2c7bb6'   # baseline blue
C_MLP     = '#1a9641'   # MLP green
C_MINHASH = '#d7191c'   # MinHash red
C_IDEAL   = '#aaaaaa'   # ideal gray
C_ADAPT   = '#ff7f00'   # adaptive orange

Saving plots to: /raid/ruban/hpmlproj/term_project


## Data (Tables 2–5 from paper)

In [2]:
# ── Table 2: 10k scale ────────────────────────────────────────────────────────
t2 = [
    # label                      R@10    R@50    R@100   QPS     VecMB   IdxMB   Build
    ('Baseline',                 .9966,  .9986,  .9990,   246,   564.0,   20.0,  51.7),
    ('MLP cosine\n(no rerank)',  .6655,  .8080,  .8464, 30150,    15.6,   44.8,   0.2),
    ('MLP K=100\n+WJ CPU',       .9946,  .9581,  .8464,   406,    15.6,   44.8,   0.2),
    ('MLP K=200\n+WJ CPU',       .9962,  .9933,  .9720,   161,    15.6,   44.8,   0.2),
    ('MLP K=500\n+WJ GPU',       .9966,  .9984,  .9980,  2904,    15.6,   32.8,   0.2),
    ('MLP K=1000\n+WJ GPU',      .9966,  .9985,  .9987,  1766,    15.6,   32.8,   0.2),
    ('Neural MinHash\n(no rerank)',.6586,.7964,  .8461, 18605,    15.6,   16.6,   0.3),
]

# ── Table 3: full 233k scale ──────────────────────────────────────────────────
t3 = [
    # label                      R@10    R@50    R@100   QPS     VecMB    IdxMB   Build
    ('Baseline',                 .9925,  .9953,  .9963,   611, 12999.0, 11519.0,  560),
    ('MLP cosine\n(no rerank)',  .6581,  .7264,  .7398,  2289,   365.0,   578.0,  124),
    ('MLP K=200\n+WJ CPU',       .9917,  .9805,  .9258,   155,   365.0,   578.0,  124),
    ('MLP K=1000\n+WJ GPU',      .9927,  .9952,  .9949,   986,   365.0,   578.0,  124),
    ('MLP K=2000\n+WJ GPU',      .9927,  .9952,  .9951,   584,   365.0,   578.0,  124),
    ('MinHash\n(no rerank)',      .5821,  .6604,  .6729,  4504,   365.0,   193.0,  131),
    ('MinHash K=500\n+WJ GPU',   .9916,  .9861,  .9648,  1345,   365.0,   193.0,  131),
    ('MinHash K=1000\n+WJ GPU',  .9922,  .9920,  .9841,   957,   365.0,   193.0,  131),
]

# ── Table 5: fixed-K vs adaptive-K at 10k ────────────────────────────────────
t5 = [
    # label             R@10    R@50    R@100   avg_K
    ('K=100 fixed',    .9963,  .9724,  .8641,  100.0),
    ('K=200 fixed',    .9966,  .9974,  .9861,  200.0),
    ('K=500 fixed',    .9966,  .9986,  .9989,  500.0),
    ('K=1000 fixed',   .9966,  .9986,  .9989, 1000.0),
    ('Adaptive\n(gap100)', .9966, .9976, .9904, 280.1),
]

# Helpers
def col(table, idx):
    return [row[idx] for row in table]

print("Data loaded.")

Data loaded.


## Figure 1 — QPS Comparison (Full Scale)

In [3]:
labels = col(t3, 0)
qps    = col(t3, 4)

colors = [
    C_BASE,    # Baseline
    C_MLP,     # MLP cosine no rerank
    C_MLP,     # MLP K=200 CPU
    C_MLP,     # MLP K=1000 GPU
    C_MLP,     # MLP K=2000 GPU
    C_MINHASH, # MinHash no rerank
    C_MINHASH, # MinHash K=500
    C_MINHASH, # MinHash K=1000
]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
bars = ax.bar(x, qps, color=colors, edgecolor='white', linewidth=0.5, width=0.65)

for bar, v in zip(bars, qps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{v:,.0f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('QPS (queries per second, 32 threads)')
ax.set_title('Query Throughput Comparison — Full 233k Scale', fontweight='bold')
ax.set_ylim(0, max(qps) * 1.18)
ax.grid(axis='y', alpha=0.3)
ax.axhline(611, color=C_BASE, linestyle='--', linewidth=1.2, alpha=0.6,
           label='Baseline QPS (611)')

legend_patches = [
    mpatches.Patch(color=C_BASE,    label='Baseline (WJ HNSW)'),
    mpatches.Patch(color=C_MLP,     label='MLP Compressor'),
    mpatches.Patch(color=C_MINHASH, label='Neural MinHash'),
]
ax.legend(handles=legend_patches, loc='upper right')
plt.tight_layout()
p = OUT / 'fig1_qps_comparison.png'
plt.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /raid/ruban/hpmlproj/term_project/fig1_qps_comparison.png


## Figure 2 — Recall@k vs Reranking Depth K

In [4]:
# MLP pipeline at full scale: K vs recall
# Rows: no-rerank (K=0 equivalent), K=200, K=1000, K=2000
mlp_k      = [0,    200,   1000,  2000]
mlp_r10    = [.6581, .9917, .9927, .9927]
mlp_r50    = [.7264, .9805, .9952, .9952]
mlp_r100   = [.7398, .9258, .9949, .9951]

minhash_k    = [0,    500,   1000]
minhash_r10  = [.5821, .9916, .9922]
minhash_r50  = [.6604, .9861, .9920]
minhash_r100 = [.6729, .9648, .9841]

baseline_r10  = 0.9925
baseline_r50  = 0.9953
baseline_r100 = 0.9963

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=False)
ks   = [10, 50, 100]
mlp_recalls  = [mlp_r10,     mlp_r50,     mlp_r100]
mh_recalls   = [minhash_r10, minhash_r50, minhash_r100]
base_recalls = [baseline_r10, baseline_r50, baseline_r100]

for ax, k, mlp_r, mh_r, base_r in zip(axes, ks, mlp_recalls, mh_recalls, base_recalls):
    ax.plot(mlp_k, mlp_r, 'o-', color=C_MLP, lw=2, ms=7, label='MLP + WJ GPU')
    ax.plot(minhash_k, mh_r, 's-', color=C_MINHASH, lw=2, ms=7, label='MinHash + WJ GPU')
    ax.axhline(base_r, color=C_BASE, linestyle='--', lw=1.5, label=f'Baseline ({base_r:.4f})')
    ax.set_xlabel('Reranking Candidate Depth K')
    ax.set_ylabel(f'Recall@{k}')
    ax.set_title(f'Recall@{k} vs K — Full 233k Scale', fontweight='bold')
    ax.set_ylim(0.55, 1.01)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    # annotate no-rerank point
    ax.annotate('no\nrerank', (0, mlp_r[0]),
                textcoords='offset points', xytext=(6, -18),
                fontsize=8, color=C_MLP)

plt.tight_layout()
p = OUT / 'fig2_recall_vs_k.png'
plt.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /raid/ruban/hpmlproj/term_project/fig2_recall_vs_k.png


## Figure 3 — Memory Footprint Comparison (Full Scale)

In [5]:
# Methods: Baseline, MLP K=1000, MinHash K=1000
mem_labels = ['Baseline\n(WJ HNSW)', 'MLP K=1000\n+WJ GPU', 'MinHash K=1000\n+WJ GPU']
vec_mb     = [12999, 365, 365]
idx_mb     = [11519, 578, 193]
build_s    = [560,   124, 131]

x = np.arange(len(mem_labels))
w = 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# Stacked memory bar
b1 = ax1.bar(x, vec_mb, width=w, label='Vector Memory', color='#4393c3', edgecolor='white')
b2 = ax1.bar(x, idx_mb, width=w, bottom=vec_mb, label='Index Memory',
             color='#2166ac', edgecolor='white')

totals = [v + i for v, i in zip(vec_mb, idx_mb)]
for xi, (v, idx, total) in enumerate(zip(vec_mb, idx_mb, totals)):
    ax1.text(xi, total + 200, f'{total/1024:.1f} GB', ha='center',
             fontsize=10, fontweight='bold')
    # gain annotation for non-baseline
    if xi > 0:
        gain = totals[0] / total
        ax1.text(xi, total / 2, f'{gain:.1f}×\nsmaller', ha='center',
                 fontsize=9, color='white', fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(mem_labels)
ax1.set_ylabel('Memory (MB)')
ax1.set_title('Total Memory Footprint — Full 233k Scale', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, max(totals) * 1.15)

# Build time bar
bar_colors = [C_BASE, C_MLP, C_MINHASH]
bars = ax2.bar(x, build_s, width=w, color=bar_colors, edgecolor='white')
for bar, v in zip(bars, build_s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{v}s', ha='center', fontsize=10, fontweight='bold')
# speedup labels
for xi, v in enumerate(build_s):
    if xi > 0:
        gain = build_s[0] / v
        ax2.text(xi, v / 2, f'{gain:.1f}×\nfaster', ha='center',
                 fontsize=9, color='white', fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(mem_labels)
ax2.set_ylabel('Index Build Time (seconds)')
ax2.set_title('Index Build Time — Full 233k Scale', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, max(build_s) * 1.15)

plt.tight_layout()
p = OUT / 'fig3_memory_comparison.png'
plt.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /raid/ruban/hpmlproj/term_project/fig3_memory_comparison.png


## Figure 4 — Recall@100 vs QPS Tradeoff (Full Scale)

In [6]:
# Each point is a (QPS, Recall@100) operating point
points = [
    # label                        QPS    R@100   color       marker  size
    ('Baseline',                   611,   .9963,  C_BASE,     '*',    180),
    ('MLP cosine\n(no rerank)',   2289,   .7398,  C_MLP,      'o',    80),
    ('MLP K=200\n+WJ CPU',        155,   .9258,  C_MLP,      'o',    80),
    ('MLP K=1000\n+WJ GPU',       986,   .9949,  C_MLP,      'D',    100),
    ('MLP K=2000\n+WJ GPU',       584,   .9951,  C_MLP,      'D',    80),
    ('MinHash\n(no rerank)',      4504,   .6729,  C_MINHASH,  'o',    80),
    ('MinHash K=500\n+WJ GPU',   1345,   .9648,  C_MINHASH,  's',    80),
    ('MinHash K=1000\n+WJ GPU',   957,   .9841,  C_MINHASH,  's',    100),
]

fig, ax = plt.subplots(figsize=(9, 6))

for label, qps, r100, color, marker, size in points:
    ax.scatter(qps, r100, c=color, marker=marker, s=size, zorder=5,
               edgecolors='white', linewidths=0.5)
    # offset labels to avoid overlap
    offsets = {
        'Baseline':              ( 10, -14),
        'MLP cosine\n(no rerank)':( 8,   6),
        'MLP K=200\n+WJ CPU':   (-10, -16),
        'MLP K=1000\n+WJ GPU':  ( 10,   5),
        'MLP K=2000\n+WJ GPU':  (-80,  -16),
        'MinHash\n(no rerank)':  (  8,   5),
        'MinHash K=500\n+WJ GPU':( 8,   5),
        'MinHash K=1000\n+WJ GPU':(-130, -16),
    }
    ox, oy = offsets.get(label, (6, 6))
    ax.annotate(label.replace('\n', ' '), (qps, r100),
                textcoords='offset points', xytext=(ox, oy),
                fontsize=8, color=color)

# Pareto frontier (MLP: no-rerank → K=1000)
pareto_qps = [2289, 986]
pareto_r   = [.7398, .9949]
ax.plot(pareto_qps, pareto_r, '--', color=C_MLP, lw=1, alpha=0.5)

ax.axhline(0.9963, color=C_BASE, linestyle=':', lw=1.2, alpha=0.7,
           label='Baseline R@100 (0.9963)')
ax.axvline(611,    color=C_BASE, linestyle=':', lw=1.2, alpha=0.7,
           label='Baseline QPS (611)')

legend_patches = [
    mpatches.Patch(color=C_BASE,    label='Baseline (WJ HNSW)'),
    mpatches.Patch(color=C_MLP,     label='MLP Compressor'),
    mpatches.Patch(color=C_MINHASH, label='Neural MinHash'),
]
ax.legend(handles=legend_patches, loc='lower right')

ax.set_xlabel('QPS (queries per second, 32 threads)')
ax.set_ylabel('Recall@100')
ax.set_title('Recall@100 vs QPS Tradeoff — Full 233k Scale\n'
             '(upper-right is better)', fontweight='bold')
ax.set_ylim(0.62, 1.005)
ax.grid(True, alpha=0.25)

plt.tight_layout()
p = OUT / 'fig4_recall_qps_tradeoff.png'
plt.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /raid/ruban/hpmlproj/term_project/fig4_recall_qps_tradeoff.png


## Figure 5 — Adaptive-K: Recall vs Average Candidate Budget

In [7]:
labels  = [row[0] for row in t5]
r10     = [row[1] for row in t5]
r50     = [row[2] for row in t5]
r100    = [row[3] for row in t5]
avg_k   = [row[4] for row in t5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: recall curves vs average K
# Fixed-K points only (first 4), then adaptive as special marker
fixed_k   = avg_k[:4]
adapt_k   = avg_k[4]

ax1.plot(fixed_k, r10[:4],  'o-', color='#1f77b4', lw=2, ms=7, label='R@10 (fixed K)')
ax1.plot(fixed_k, r50[:4],  's-', color='#ff7f0e', lw=2, ms=7, label='R@50 (fixed K)')
ax1.plot(fixed_k, r100[:4], '^-', color='#2ca02c', lw=2, ms=7, label='R@100 (fixed K)')

# Adaptive-K as stars on each curve
ax1.scatter([adapt_k], [r10[4]],  marker='*', s=200, color='#1f77b4', zorder=6)
ax1.scatter([adapt_k], [r50[4]],  marker='*', s=200, color='#ff7f0e', zorder=6)
ax1.scatter([adapt_k], [r100[4]], marker='*', s=200, color='#2ca02c', zorder=6)

ax1.axvline(adapt_k, color=C_ADAPT, linestyle='--', lw=1.5,
            label=f'Adaptive-K avg = {adapt_k:.0f}')
ax1.set_xlabel('Average Candidate Budget K')
ax1.set_ylabel('Recall')
ax1.set_title('Recall vs Candidate Budget K — 10k Scale\n'
              '(★ = Adaptive-K operating point)', fontweight='bold')
ax1.set_ylim(0.84, 1.005)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Right: K distribution pie chart for adaptive-K
k_buckets = ['K=100\n(9.1%)', 'K=200\n(69.9%)', 'K=500\n(15.9%)', 'K=1000\n(5.2%)']
k_sizes   = [9.1, 69.9, 15.9, 5.2 - 0.1]   # small rounding fix
k_sizes   = [9.1, 69.9, 15.9, 5.1]
k_colors  = ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c']
explode   = (0, 0.05, 0, 0)

wedges, texts, autotexts = ax2.pie(
    k_sizes, labels=k_buckets, colors=k_colors,
    autopct='%1.1f%%', startangle=140, explode=explode,
    textprops={'fontsize': 9}
)
for at in autotexts:
    at.set_fontsize(8.5)
ax2.set_title(f'Adaptive-K Budget Distribution\n'
              f'(avg K = {adapt_k:.0f} vs fixed K=1000 → 3.6× cost reduction)',
              fontweight='bold')

plt.tight_layout()
p = OUT / 'fig5_adaptive_k.png'
plt.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /raid/ruban/hpmlproj/term_project/fig5_adaptive_k.png


## Summary

In [8]:
figs = [
    ('fig1_qps_comparison.png',   'QPS bar chart — full 233k scale'),
    ('fig2_recall_vs_k.png',      'Recall@10/50/100 vs reranking depth K'),
    ('fig3_memory_comparison.png','Memory footprint + build time comparison'),
    ('fig4_recall_qps_tradeoff.png','Recall@100 vs QPS scatter (Pareto)'),
    ('fig5_adaptive_k.png',       'Adaptive-K recall curves + budget distribution'),
]
print(f"All figures saved to: {OUT}\n")
for fname, desc in figs:
    exists = '✓' if (OUT / fname).exists() else '✗'
    print(f"  {exists}  {fname:<38}  {desc}")

All figures saved to: /raid/ruban/hpmlproj/term_project

  ✓  fig1_qps_comparison.png                 QPS bar chart — full 233k scale
  ✓  fig2_recall_vs_k.png                    Recall@10/50/100 vs reranking depth K
  ✓  fig3_memory_comparison.png              Memory footprint + build time comparison
  ✓  fig4_recall_qps_tradeoff.png            Recall@100 vs QPS scatter (Pareto)
  ✓  fig5_adaptive_k.png                     Adaptive-K recall curves + budget distribution
